# 📊 Enterprise Gemini Stateless Batch Transcription Pipeline

This notebook orchestrates a high-performance, serverless-native batch transcription pipeline across tactical radio dispatch channels. It implements our production-grade **Stateless Audio-Only Rolling History (Path B)** architecture using the new `google-genai` SDK.

### 🚀 Key Architectural Breakthroughs:
1. **0.00% Hallucinations (Audio-Only History):** By completely removing noisy past text transcripts from the lookback context and passing only raw GCS audio files, we eliminate the feedback loop of error propagation.
2. **Serverless & Database-Free:** Bypasses Spanner, session locks, and Reasoning Engines. The history is maintained as a simple, stateless list of GCS URIs, making it 100% horizontally scalable and 90% cheaper.
3. **Blazing-Fast Latency:** Executes in a flat, predictable approx. 8-15 seconds, representing a 2x to 4x speedup over the legacy stateful session approach.


In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present (for hosted Colab environments)
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git

# Install the model library in editable mode along with required dependencies
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model loguru tqdm

    import site
    import importlib

    importlib.reload(site)
    print("\n✅ Dependencies installed successfully.")

In [ ]:
# @title Imports
import asyncio
import collections
import json
import os
import re
import sys
import time
from collections import defaultdict, deque
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.api_core.exceptions import ClientError, GoogleAPICallError
from google.api_core import retry as api_retry
from google.api_core import retry_async as api_retry_async
from google.colab import auth, userdata
from loguru import logger
from tqdm.asyncio import tqdm

# Third-party GenAI imports
from google.genai import types, Client as GenAiClient

In [ ]:
# @title Authenticate with GCP
# @markdown Run this cell to authenticate your browser session with Google Cloud.
print("Attempting standard browser authentication...")
auth.authenticate_user()
print("✅ Browser session authenticated successfully.")

In [ ]:
# @title Pipeline Configuration

# @markdown ### Model Selection
# @markdown **Option A: Select a base model**
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}

# @markdown **Option B: Use a Tuned ML Model (Deployed in multi-region us)**
USE_CUSTOM_MODEL = False  # @param {type:"boolean"}
CUSTOM_ENDPOINT_ID = ""  # @param {type:"string"}

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCP_PROJECT_NUMBER = userdata.get("GCP_PROJECT_NUMBER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
if not CUSTOM_ENDPOINT_ID:
    CUSTOM_ENDPOINT_ID = userdata.get("CUSTOM_ENDPOINT_ID")

# @markdown ### GCP Infrastructure Configuration
GCP_LOCATION = "us"  # @param {type:"string"}

# @markdown ### Input/Output Configuration
# @markdown Partial path under `gs://{GCS_BUCKET}/segmented_audio/` where `batch_manifest.jsonl` and audio segments are located.
# fmt: off
INPUT_AUDIO_DIR = "broadcastify/calls/eval_audio_masked_v2"  # @param {type:"string"}
# fmt: on
# @markdown Partial path under `gs://{GCS_BUCKET}/transcripts/` where outputs will be stored.
OUTPUT_TRANSCRIPT_DIR = "broadcastify/calls/eval"  # @param {type:"string"}
# @markdown Experiment Name (representing the subfolder under the model directory, e.g., `bcfy_calls_v1`).
EXPERIMENT_NAME = "bcfy_calls_v1"  # @param {type:"string"}

# @markdown ### Stateless Context Window Settings
# @markdown Capped lookback context size (total number of parts in the prompt).
# @markdown Size 11 represents 10 past turns (approx. 2 minutes of audio). Size 21 represents 20 past turns (approx. 4 minutes).
CONTEXT_SIZE = 21  # @param {type:"integer"}
CONCURRENCY_LIMIT = 5  # @param {type:"integer"}
OVERWRITE_EXISTING = True  # @param {type:"boolean"}

assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert OUTPUT_TRANSCRIPT_DIR, (
    "OUTPUT_TRANSCRIPT_DIR must be provided and cannot be empty."
)
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."

# Streamlined Output Path Generation
if USE_CUSTOM_MODEL:
    assert CUSTOM_ENDPOINT_ID, (
        "CUSTOM_ENDPOINT_ID must be provided as a parameter or in Colab secrets (userdata)."
    )
    if CUSTOM_ENDPOINT_ID.startswith("projects/"):
        MODEL_PATH = CUSTOM_ENDPOINT_ID
    else:
        proj_handle = GCP_PROJECT_NUMBER or GCP_PROJECT_ID
        assert proj_handle, (
            "GCP_PROJECT_NUMBER or GCP_PROJECT_ID must be provided to construct endpoint path."
        )
        target_loc = GCP_LOCATION if GCP_LOCATION != "global" else "us"
        MODEL_PATH = f"projects/{proj_handle}/locations/{target_loc}/endpoints/{CUSTOM_ENDPOINT_ID}"
    MODEL_ID_DIR = re.sub(r"[-\.]", "_", CUSTOM_ENDPOINT_ID.split("/")[-1])
else:
    MODEL_PATH = f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}"
    MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)

GCS_OUTPUT_BASE = (
    f"transcripts/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/segmented_audio/{INPUT_AUDIO_DIR}/batch_manifest.jsonl"
)
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"
CHECKPOINT_FILE = "interim_backup_predictions.jsonl"

SYSTEM_PROMPT = """\
Your primary task is to produce a strict, verbatim transcription of the spoken audio. Your absolute highest priority is to transcribe only what you hear with high acoustic certainty. Do not add, invent, or infer any speech that is not clearly audible. The audio may originate from VHF/UHF radio traffic and can include mic clicks, RF static, radio hum, and potentially unintelligible speech. When the audio is unequivocally confirmed as fire-related dispatch, speakers often use heavy jargon, and specific formatting rules apply.

EXPECTED TERMINOLOGY:
These are terms and unit identifiers commonly used in fire-related dispatch. These terms and formatting rules apply exclusively to audio that is unequivocally confirmed as fire-related dispatch. If these exact terms are clearly heard in the audio, transcribe them as listed. Do not invent or infer the use of these terms if they are not genuinely spoken.
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript strictly and precisely as spoken in the audio, with no newlines. Do not add, invent, or infer any speech that is not clearly audible.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. If the audio contains a unit identifier, format it as the unit type followed by digits (e.g., Engine 41, Battalion 2). Apply this rule strictly only if the unit identifier is clearly spoken AND the context is unequivocally fire-related dispatch.
4. Transcribe only the duration of speech present. Do not extend the transcription with additional words or phrases that were not spoken, even if contextually plausible.

QUALITY GATE: Your absolute highest priority is to transcribe only what you hear with high acoustic certainty.
    *   If the audio contains clear speech that is not fire-related dispatch, you MUST transcribe it verbatim, exactly as heard, without applying any fire-specific formatting or jargon, and without attempting to interpret it as fire dispatch traffic.
    *   If a portion of audio is obscured, noisy, ambiguous, or contains speech that cannot be confidently identified, you MUST replace that specific portion with [UNINTELLIGIBLE].
    *   Do not attempt to infer, guess, or invent speech to fit any expected context or terminology list.
    *   Do not attempt to phonetically guess ambiguous noise.
    *   If the entire audio segment does not contain any discernible speech, output only [UNINTELLIGIBLE].

TASK:
You will receive a sequence of audio files representing the history of the channel. The final audio file in the list is the new segment. Use the preceding audio files ONLY as acoustic and vocabulary context reference. Transcribe ONLY the final audio file verbatim. Output strictly the transcript of the final segment.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
    "thinking_config": types.ThinkingConfig(thinking_budget=0),
}

logger.remove()
_ = logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="INFO"
)

In [ ]:
# @title Pipeline Execution Logic

# Import the production-grade, unpolluted stateless rolling transcription engine
# from your repository's dedicated 'common.gemini' package!
from common.gemini.stateless import process_single_channel_stateless

In [ ]:
# @title Execute Batch Transcriptions

# Shared locks and GCS client setup
checkpoint_write_lock = asyncio.Lock()
storage_client = storage.Client(project=GCP_PROJECT_ID)


def get_gcs_checkpoint_blob():
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    checkpoint_blob_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE
    return storage_client.bucket(out_bucket).blob(checkpoint_blob_path)


def load_gcs_checkpoint() -> dict[str, str]:
    blob = get_gcs_checkpoint_blob()
    records = {}
    if blob.exists():
        logger.info(
            f"Found existing checkpoint on GCS: {blob.name}. Loading..."
        )
        blob.download_to_filename(CHECKPOINT_FILE)
        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record["transcript"]
        logger.info(
            f"Loaded {len(records)} completed records from GCS checkpoint."
        )
    else:
        logger.info("No remote checkpoint found. Starting fresh.")
    return records


async def main() -> None:
    logger.info("Initializing Google Cloud and GenAI clients...")

    genai_client = GenAiClient(
        project=GCP_PROJECT_ID,
        location=GCP_LOCATION,
        vertexai=True,
    )

    completed_records = {}
    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs.")
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
    else:
        completed_records = load_gcs_checkpoint()

    # Download and parse manifest
    logger.info(f"Downloading manifest from GCS: {MANIFEST_URI}...")
    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)

    if not manifest_blob.exists():
        raise FileNotFoundError(
            f"Manifest GCS file not found at {MANIFEST_URI}"
        )

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0
    for line in content:
        if line.strip():
            entry = json.loads(line)
            channel_id = entry.get("source_group")
            if not channel_id:
                eid = entry.get("example_id")
                if eid and "-row-" not in str(eid):
                    channel_id = eid
            if not channel_id:
                channel_id = Path(entry["audio_filepath"]).parent.name
            entry["example_id"] = channel_id
            channels[channel_id].append(entry)
            total_segments += 1

    for ch in channels:
        channels[ch].sort(
            key=lambda x: (
                x.get("original_audio_uri")
                or x.get("audio_uri")
                or x.get("audio_filepath", ""),
                x.get("original_offset")
                if x.get("original_offset") is not None
                else x.get("offset", 0),
                x.get("start_time", 0),
            )
        )

    # Filter for what is actually missing
    active_channels = {}
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries

    if not active_channels:
        logger.info("🎉 All transcriptions are already complete!")
    else:
        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        logger.info(
            f"Starting stateless batch run across {len(active_channels)} channels..."
        )

        with tqdm(
            total=total_segments, desc="Stateless Transcriptions"
        ) as pbar:
            loop = asyncio.get_running_loop()
            tasks = [
                loop.create_task(
                    process_single_channel_stateless(
                        channel_id=cid,
                        segments=entries,
                        completed_records=completed_records,
                        genai_client=genai_client,
                        model_path=MODEL_PATH,
                        system_prompt=SYSTEM_PROMPT,
                        safety_settings=SAFETY_SETTINGS,
                        generation_config=GENERATION_CONFIG,
                        context_size=CONTEXT_SIZE,
                        concurrency_limit=CONCURRENCY_LIMIT,
                        checkpoint_file=CHECKPOINT_FILE,
                        checkpoint_write_lock=checkpoint_write_lock,
                        semaphore=semaphore,
                        pbar=pbar,
                    )
                )
                for cid, entries in active_channels.items()
            ]
            try:
                await asyncio.gather(*tasks)
            except Exception as e:
                logger.error(
                    f"FATAL Exception detected: {e!s}. Cancelling all other active channels..."
                )
                for t in tasks:
                    if not t.done():
                        t.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
                raise

    # Upload final merged ndjson predictions to GCS
    if os.path.exists(CHECKPOINT_FILE):
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_filename(
            CHECKPOINT_FILE
        )
        logger.success(
            f"🎉 Batch execution complete! Predictions saved to: {CONSISTENT_OUTPUT_URI}"
        )

        # Clean up local checkpoint file
        os.remove(CHECKPOINT_FILE)


await main()

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

# Parse outputs to count failures
failures = []
for line in output_content:
    if line.strip():
        record = json.loads(line)
        if record.get("error"):
            failures.append(record)

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")
print(f"Failed Segments (Errors):      {len(failures)}")

if expected_count == actual_count:
    if failures:
        print(
            f"\n⚠️ WARNING: All segments were recorded, but {len(failures)} segments FAILED with errors!"
        )
        print("Please re-run the pipeline cell to retry the failures.")
    else:
        print(
            "\n✅ SUCCESS: All segments were transcribed successfully with zero errors!"
        )
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )

In [ ]:
# @title Channel Isolation & Channel ID Audit


def audit_manifest_isolation():
    print(f"--- Auditing Manifest: {MANIFEST_URI} ---\n")

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(m_bucket)
        .blob(m_path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    channel_to_paths = defaultdict(set)
    path_to_channels = defaultdict(set)

    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        cid = entry.get("example_id")
        path = entry.get("audio_filepath")

        # Extract the source folder name from the path as a 'ground truth' source
        source_folder = path.split("/")[-2] if "/" in path else "unknown"

        channel_to_paths[cid].add(source_folder)
        path_to_channels[source_folder].add(cid)

    # Check 1: Does one channel ID map to multiple physical folders? (Channel Pollution)
    overlap_found = False
    for cid, sources in channel_to_paths.items():
        if len(sources) > 1:
            print(
                f"⚠️ COLLISION: Channel ID '{cid}' is being shared by multiple sources: {sources}"
            )
            print(
                "   This WILL cause context pollution and out-of-order histories."
            )
            overlap_found = True

    # Check 2: Does one physical folder have multiple channel IDs?
    for source, cids in path_to_channels.items():
        if len(cids) > 1:
            print(
                f"ℹ️ Note: Source '{source}' is split across multiple IDs: {cids}"
            )

    if not overlap_found:
        print(
            "✅ Isolation Audit Passed: Each Channel ID maps to exactly one source directory."
        )

    print(f"\nUnique Channels to be transcribed: {len(channel_to_paths)}")


audit_manifest_isolation()